# all-reduce-eval-metrics — faded example 1: Finish the global accuracy reduce: the final divide

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`. The last cell reports your progress on the `Distributed: all_reduce eval metrics` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Per-rank `(correct, seen)` counts must be summed globally and divided once to get true accuracy over the whole test set. The reduce gives you total correct in `stats[0]` and total seen in `stats[1]`; the last step turns those raw totals into the accuracy scalar. Dividing per-sample totals (not averaging per-rank accuracies) is what handles ragged shards correctly.

## Faded exercise 1

The packing and the `all_reduce(SUM)` are written for you: after the reduce, `stats[0]` holds total correct and `stats[1]` holds total samples across all ranks. Complete the one line that turns those two synchronized totals into the global top-1 accuracy returned as a Python float.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def global_accuracy(dist_module, local_correct, local_seen):
    stats = t.tensor([float(local_correct), float(local_seen)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    acc = None  # TODO: fill in this step — read the prompt cell above
    return acc

t.manual_seed(0)
correct = [30, 36, 28, 8]
seen = [40, 40, 40, 8]
rank_tensors = [t.tensor([float(c), float(s)], dtype=t.float32) for c, s in zip(correct, seen)]
mock = MockDist(rank_tensors)
result = global_accuracy(mock, correct[0], seen[0])
print('global accuracy:', round(result, 6))


def _test():
    correct = [30, 36, 28, 8]
    seen = [40, 40, 40, 8]
    expected = sum(correct) / sum(seen)
    rank_tensors = [t.tensor([float(c), float(s)], dtype=t.float32) for c, s in zip(correct, seen)]
    mock = MockDist(rank_tensors)
    got = global_accuracy(mock, correct[0], seen[0])
    assert isinstance(got, float), f'expected python float, got {type(got)}'
    assert abs(got - expected) < 1e-6, f'got {got}, expected {expected}'
    assert abs(got - 0.796875) < 1e-6, f'got {got}, expected 0.796875'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def global_accuracy(dist_module, local_correct, local_seen):
    stats = t.tensor([float(local_correct), float(local_seen)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    acc = (stats[0] / stats[1]).item()
    return acc

t.manual_seed(0)
correct = [30, 36, 28, 8]
seen = [40, 40, 40, 8]
rank_tensors = [t.tensor([float(c), float(s)], dtype=t.float32) for c, s in zip(correct, seen)]
mock = MockDist(rank_tensors)
result = global_accuracy(mock, correct[0], seen[0])
print('global accuracy:', round(result, 6))
```
</details>